# Module 2 · Lecture 1 — Classification with a CNN (ResNet)

**Course:** Deep Learning for Medical Image Analysis.  
This notebook is the hands-on companion to the *Lecture 1* slides. We train a
**ResNet** to classify medical images, following the *universal training recipe*:

> `Dataset → DataLoader → Model → Loss → Optimizer → loop → evaluate`

**Dataset:** [PathMNIST](https://medmnist.com/) — 9-class colon-pathology tiles
(RGB, 28×28), part of MedMNIST v2. Small and pip-installable, so it runs on a
**free Colab GPU**.

> **Runtime:** set `Runtime → Change runtime type → GPU` before running.

> `QUICK_RUN = True` (below) trains for 1 epoch on a subset so the whole notebook
> finishes in ~1 minute — a smoke test. Set it to `False` for a real run.

## 0 · Setup

In [ ]:
# Colab: install the two extra packages (torch/torchvision are preinstalled on Colab).
%pip install -q medmnist

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as T
from torchvision import models
import matplotlib.pyplot as plt

import medmnist
from medmnist import INFO

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Smoke-test switch: 1 epoch on a subset for a fast end-to-end check.
QUICK_RUN = True

## 1 · Data — PathMNIST

We resize the 28×28 tiles to 64×64 so a standard ResNet stem behaves sensibly,
and normalize with ImageNet statistics (the pretrained backbone expects them).

MedMNIST labels come as shape `(N, 1)`; we squeeze them to a 1-D `Long` tensor
for `CrossEntropyLoss`.

In [ ]:
data_flag = 'pathmnist'
info = INFO[data_flag]
n_channels = info['n_channels']
n_classes  = len(info['label'])
class_names = [info['label'][str(i)] for i in range(n_classes)]
DataClass = getattr(medmnist, info['python_class'])
print(f'{data_flag}: {n_channels} channels, {n_classes} classes')
print('classes:', class_names)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

transform = T.Compose([
    T.Resize(64),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_ds = DataClass(split='train', transform=transform, download=True)
val_ds   = DataClass(split='val',   transform=transform, download=True)
test_ds  = DataClass(split='test',  transform=transform, download=True)
print('train/val/test:', len(train_ds), len(val_ds), len(test_ds))

In [ ]:
def collate(batch):
    xs, ys = zip(*batch)
    x = torch.stack(xs)
    y = torch.tensor([int(np.array(t).squeeze()) for t in ys], dtype=torch.long)
    return x, y

BATCH = 128
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, collate_fn=collate)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, collate_fn=collate)

xb, yb = next(iter(train_loader))
print('batch:', xb.shape, yb.shape, yb[:8].tolist())

### Export a sample montage for the slides

Saves `../Figures/pathmnist_samples.png`, which the Lecture-1 *Hands-on* slide
picks up automatically (`\IfFileExists`).

In [ ]:
def denorm(img):
    img = img.numpy().transpose(1, 2, 0)
    img = img * np.array(IMAGENET_STD) + np.array(IMAGENET_MEAN)
    return np.clip(img, 0, 1)

fig, axes = plt.subplots(3, 6, figsize=(6, 3))
for ax, img, lab in zip(axes.ravel(), xb, yb):
    ax.imshow(denorm(img)); ax.set_title(class_names[lab][:10], fontsize=5)
    ax.axis('off')
plt.tight_layout()

import os
fig_dir = os.path.join('..', 'Figures')
os.makedirs(fig_dir, exist_ok=True)
out_png = os.path.join(fig_dir, 'pathmnist_samples.png')
plt.savefig(out_png, dpi=150, bbox_inches='tight', facecolor='white')
print('saved', out_png)
plt.show()

## 2 · Model — ResNet-18

Two ways to get the classifier `f(x; W)` from the slides:

1. **From scratch** — random init, learns everything from PathMNIST.
2. **Pretrained + fine-tune** — start from ImageNet features and adapt the final
   fully-connected layer to our 9 classes. Usually wins on small medical data
   (Module 1: *data scarcity*).

In [ ]:
def build_resnet18(pretrained: bool, n_classes: int):
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, n_classes)  # new head
    return model

# Switch to compare: pretrained=True (fine-tune) vs pretrained=False (from scratch).
model = build_resnet18(pretrained=True, n_classes=n_classes).to(device)
print(sum(p.numel() for p in model.parameters())/1e6, 'M parameters')

## 3 · Train

This is the *universal recipe* from the slides, verbatim:
`zero_grad → forward → loss → backward → step`.

In [ ]:
criterion = nn.CrossEntropyLoss()          # softmax + NLL (Module 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def run_epoch(loader, train: bool):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):          # scoped: restores state on exit
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            loss_sum += loss.item() * x.size(0)
            correct  += (logits.argmax(1) == y).sum().item()
            total    += x.size(0)
    return loss_sum/total, correct/total

In [ ]:
EPOCHS = 1 if QUICK_RUN else 5

# QUICK_RUN: train on a small subset so the smoke test finishes fast.
if QUICK_RUN:
    idx = torch.randperm(len(train_ds))[:2000]
    quick_loader = DataLoader(Subset(train_ds, idx.tolist()), batch_size=BATCH,
                              shuffle=True, collate_fn=collate)
    fit_loader = quick_loader
else:
    fit_loader = train_loader

for epoch in range(EPOCHS):
    tr_loss, tr_acc = run_epoch(fit_loader, train=True)
    va_loss, va_acc = run_epoch(val_loader, train=False)
    print(f'epoch {epoch+1}/{EPOCHS}  '
          f'train_loss={tr_loss:.3f} acc={tr_acc:.3f}  |  '
          f'val_loss={va_loss:.3f} acc={va_acc:.3f}')

## 4 · Evaluate

Accuracy alone can mislead on imbalanced medical data (slides). We also look at
the **confusion matrix** and a **macro one-vs-rest AUC**.

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score, accuracy_score

model.eval()
all_logits, all_y = [], []
with torch.no_grad():
    for x, y in test_loader:
        all_logits.append(model(x.to(device)).cpu())
        all_y.append(y)
logits = torch.cat(all_logits); y_true = torch.cat(all_y).numpy()
probs  = torch.softmax(logits, dim=1).numpy()
y_pred = probs.argmax(1)

print('test accuracy:', round(accuracy_score(y_true, y_pred), 4))
try:
    auc = roc_auc_score(y_true, probs, multi_class='ovr', average='macro')
    print('macro OVR AUC:', round(auc, 4))
except ValueError as e:
    print('AUC skipped (a class may be missing in the quick subset):', e)

In [ ]:
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='cividis')
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title('PathMNIST confusion')
fig.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(os.path.join(fig_dir, 'pathmnist_confusion.png'), dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

## 5 · Inspect predictions

In [ ]:
xb, yb = next(iter(test_loader))
with torch.no_grad():
    pred = model(xb.to(device)).argmax(1).cpu()
fig, axes = plt.subplots(2, 6, figsize=(7, 2.6))
for ax, img, t, p in zip(axes.ravel(), xb, yb, pred):
    ax.imshow(denorm(img)); ax.axis('off')
    ok = (t == p)
    ax.set_title(f'{class_names[p][:6]}', fontsize=6,
                 color=('green' if ok else 'red'))
plt.tight_layout(); plt.show()

## Recap & next steps

- We trained a ResNet end-to-end with the universal recipe and evaluated it with
  metrics that matter clinically.
- **Try:** flip `pretrained=False` (Section 2) and `QUICK_RUN=False` — compare
  from-scratch vs fine-tuned accuracy.
- **Swap the dataset** by changing `data_flag` in Section 1:
  - `'pneumoniamnist'` — binary chest X-ray (simplest);
  - `'bloodmnist'` — 8-class blood-cell typing (ties to Module 1's cell task).

Next lecture: **segmentation** (per-pixel labels) with a **U-Net**.